## IMPORT THU VIEN ##

In [ ]:
# --- NHÓM THƯ VIỆN CỐT LÕI ---
import cv2
import numpy as np
import time
from PIL import Image
import threading

# --- NHÓM AI & DỊCH THUẬT ---
from ultralytics import YOLO          
from manga_ocr import MangaOcr        # OCR tiếng Nhật chuyên dụng
from deep_translator import GoogleTranslator # Dịch thuật

# --- NHÓM HỆ THỐNG & CAPTURE ---
import mss                            # Chụp màn hình tốc độ cực cao
from PyQt6.QtWidgets import QApplication, QWidget # Làm giao diện Overlay
from PyQt6.QtCore import Qt, QTimer
from PyQt6.QtGui import QPainter, QColor, QFont

## GÓC NHÌN BAN ĐẦU VỀ ỨNG DỤNG (Trước mắt là cho PC nhưng cứ note cho đthoai để về sau deploy nếu muốn) ##

- Mở app lên (run file, ép thành exe,...) thì sẽ có một giao diện tối thiểu gồm hai phần

1. Phần cap màn hình: Kéo thả chuột / dùng ngón tay để kéo chọn vùng muốn dịch
    -> Các hàm thư viện bắt tọa độ để biết tọa độ 4 góc (phạm vi trình dịch chạy)
    - Hiển nhiên là sau khi cap xong sẽ có một nút hay gdien khác hiện lên cho phép cap mới. Chọn cap mới thì overwrite cái dữ liệu phạm vi cũ.

1.1: Trình dịch bắt đầu hoạt động.

- Nhớ là có các thư viện có hàm cho phép cap ảnh trong một phạm vi ( hiển nhiên là nhét cái tọa độ thu được sau khi người dùng kéo thả) + tần suất 1 giây cap mấy lần.

- Ảnh cap qua quá trình đó tất nhiên vứt vào bộ nhớ tạm, và chỉ dùng 1 biến để lưu các ảnh tạm đó thôi để đỡ quá tải bộ nhớ tạm.

(Optional): Về sau nếu cần thì tự viết thêm một vài hàm chỉnh ảnh như dùng histogram, lọc mịn,...
            Do đa số manga trực tuyến đều net nên chỉ cân nhắc thôi, sợ vỡ ảnh vỡ text

Note: 'names': {0: 'body', 1: 'face', 2: 'frame', 3: 'text'}

=> Nạp cái ảnh vào con model được train để nó bắt text thui, sau đó nhét qua mangaOCR để trích xuất ra text, rồi gọi PapagoTranslator (hoặc googletrans, nói chung là API dịch free) từ deep_trans để dịch sang ngôn ngữ chỉ định

    - Cái phần chỉ định ngôn ngữ là giao diện tối thiểu thứ hai: trước mỗi phiên cap thì khi ấn vào cái nút "Dịch" thì pop up lên hai option con, bên trái là "từ", khi ấn vào lại pop-up ra một loạt ngôn ngữ, nhưng ưu tiên ở vị trí thứ 1,2,3 lần lượt là auto-detect, eng và jp

        - ở bên phải là "sang" thì ưu tiên vị trí thứ 1 là vietnamese

    - Cả hai bên chắc chỉ nhét dưới 10 ngôn ngữ nổi nhất thôi.

    - Nếu user không tạo ra sự thay đổi gì thì mặc định bên "từ" là auto detect, còn "sang" là vietnamese

    => Hiển nhiên là sau đấy nhét mấy tên đại diện ngôn ngữ (mapping) vào API dịch

        - Nhét luôn các text đã được trích xuất vào để dịch sang ngôn ngữ đã quy ước

2. Vẽ lại lên trên màn hình

- Dùng tọa độ về text mà con model trả về, lấy dư ra tầm 10-20px j đấy (không to quá, để cho chắc) rồi dùng thư viện nào đó có hàm cho phép vẽ lên các vùng tọa độ hoặc gì đấy, tóm lại là vẽ được. "vẽ" các dòng đã dịch lên.

=> Lặp lại thôi.

+Khi viết code nhớ để ý với cách lưu ảnh / dịch ảnh của mình như vậy thì việc chụp ảnh và dịch thực tế có mượt mà không? (Hạn chế hết mức có thể tình trạng kiểu: người dùng mới cuộn chuột, lướt tay hay làm gì đó di chuyển khung ảnh đã chụp thì trình dịch lại chạy lại từ đầu, khúc này có khi nhét các câu đã dịch vào bộ nhớ tạm hay gì đó chẳng hạn, xong đặt điều kiện là cho đến khi phát hiện text mới thì kể cả có dịch khung truyện thì chương trình không chạy lại cả quá trình chụp - dịch -vẽ)

-> Hiển nhiên là nên vứt lên AI để tham khảo ý kiến của nó.

# ý kiến sau khi tham khảo
- Không nên: OCR + dịch mỗi frame mà chỉ gọi quá trình khi text thay đổi
=> Viết điều kiện gì đấy kiểu: mỗi một khoảng thời gian quét lại một lần, nếu text không đổi thì không gọi dịch (để làm được thế thì có khi đưa tất cả các text gốc được cap trước khi qua workflow vào một danh sách tạm chăng? Lần đầu cap ảnh thì đưa các text vào list, còn sau gọi kiểm tra thì gọi luôn cái list đó làm điều kiện)